# Onboarding — SportsPredict Model API (Jump Trading Probability Cup)

This notebook walks through **every capability of the API**, one section per endpoint.
Full reference: `docs/API-Documentation.pdf`.

## The game

- 2026 World Cup, **Jun 11 → Jul 19 2026**. ~72 matches, each with ~10 binary yes/no markets (~720 total).
- For each market you submit a probability as an **integer 1–99** before the match kicks off
  (in the live data, a match's `opening_time` is the kickoff — see the warning in section 3).
- You may revise a prediction (PATCH) any time before the deadline — the **last value is what gets scored**.

## Scoring

| Metric | Formula | Meaning |
|---|---|---|
| Brier | $(p - o)^2$ | $o \in \{0,1\}$ is the outcome. **Lower is better**, 0 = perfect |
| RBP | $(\text{crowd brier} - \text{your brier}) \times 100$ | Positive = you beat the crowd on that market |
| Smart Rating | percentile of cumulative RBP | This is the leaderboard. Stage weights: group **1×**, knockout **2×**, final **3×** |

**Key insight:** you are scored *relative to the crowd average*, so you win by being
better-calibrated than the crowd — especially on knockout/final matches, which count 2–3×.

## API flow

```
GET /events ──> GET /lobbies ──> POST /lobbies/{id}/join     (one-time onboarding)
GET /matches ──> GET /markets?match_id=...                   (discovery, repeat as matches open)
POST /predictions/batch (≤50) ──> PATCH /predictions/{id}    (submit, then revise until kickoff)
GET /predictions ──> GET /results                            (monitor; /results has Brier scores)
```

Auth is a Bearer key (`SPORTSPREDICT_KEY_1` in `.env`). Rate limit: **60 requests/min per IP**.

## 0. Setup

All HTTP details (auth header, retries, 429 backoff, batch chunking) live in `src/probcup/sportspredict/client.py`
so they're written once and shared by every notebook and script. The notebook only *uses* the client.

In [ ]:
import pandas as pd

from probcup.sportspredict.client import Client  # `pip install -e .` (or `make setup`)

pd.set_option("display.max_colwidth", 90)

# bot=1 -> SPORTSPREDICT_KEY_1 in .env, bot=2 -> SPORTSPREDICT_KEY_2.
# Each bot is its own leaderboard entry; all onboarding here uses bot 1.
client = Client(bot=1)
print("Client ready")

## 1. Events — `GET /events`

Lists active and upcoming events. The Model API only surfaces *probability* events
(Brier-scored). One docs-vs-reality quirk: the docs say `type == "probability"`, but the
live API returns an opaque type UUID — so we identify the Probability Cup by its title
(handled inside `client.probability_cup_event()`).

In [ ]:
events = client.events()
display(pd.DataFrame(events))

# The event we are competing in — every later call hangs off this id
event = client.probability_cup_event()
print(f"Competing in: {event['title']}  ({event['start_date']} -> {event['end_date']})")

## 2. Lobby — `GET /lobbies` + `POST /lobbies/{id}/join`

The Probability Cup has **one shared lobby** that every participant joins (free, no stake).
You must be a member before you can view markets or submit predictions.
Joining is idempotent: a `409 Conflict` just means you're already in (the client treats that as success).

In [ ]:
lobby = client.lobbies(event["id"])[0]
print(f"Lobby: {lobby['name']}  (joined: {lobby['joined']})")

if not lobby["joined"]:
    client.join_lobby(lobby["id"])
    print("Joined the lobby.")

## 3. Matches — `GET /matches`

The discovery step: every match that still has at least one open market, with kickoff times.
Lightweight (~9 KB for the whole group stage).

**⚠️ Timing — the live data contradicts the docs PDF.** The docs say `closing_time` is the
match start / prediction deadline. In reality, `opening_time` is the **kickoff** (verified
against the official match schedule) and `closing_time` is consistently 2.5 h later — i.e.
roughly the match *end*. **Treat `opening_time` as the prediction deadline**: submitting
after kickoff may or may not be accepted, and relying on it would be both risky and
pointless (the crowd you're scored against could do the same).

In [ ]:
matches = client.matches(event_id=event["id"])

matches_df = (
    pd.DataFrame(matches)
    .assign(opening_time=lambda d: pd.to_datetime(d["opening_time"]))
    .sort_values("opening_time")
    .reset_index(drop=True)
)
print(f"{len(matches_df)} matches currently have open markets")
matches_df.head(15)[["name", "opening_time", "closing_time", "open_market_count", "id"]]

## 4. Markets — `GET /markets?lobby_id=&match_id=`

Each market is **one binary question** about one match. Always filter by `match_id`
(~10 markets / ~4 KB per call) — unfiltered, the group stage returns ~720 markets in one ~318 KB response.

There is no `current_price` field on probability markets: your prediction stands alone
and is scored against the eventual outcome (you can't see the crowd's number).

In [ ]:
# Look at the next upcoming match as a concrete example
next_match = matches_df.iloc[0]
markets = client.markets(lobby_id=lobby["id"], match_id=next_match["id"])

# opening_time is the kickoff — our effective prediction deadline (see section 3)
print(f"Markets for: {next_match['name']}  (kickoff {next_match['opening_time']:%Y-%m-%d %H:%M UTC})")
pd.DataFrame(markets)[["question", "status", "id"]]

### 4b. Snapshot every open market

One call per match (~72 calls ≈ within the 60/min limit thanks to the client's backoff).
We save the result to `data/cache/snapshot.json` so model work can happen offline, and
normalize the questions into **templates** — these templates are exactly what our model must predict.

In [ ]:
import json
from collections import Counter
from datetime import datetime, timezone

all_markets = []
for i, match in matches_df.iterrows():
    all_markets += client.markets(lobby_id=lobby["id"], match_id=match["id"])
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(matches_df)} matches fetched...")
print(f"{len(all_markets)} open markets in total")

data_dir = Path.cwd().parent / "data"
data_dir.mkdir(exist_ok=True)
snapshot = {
    "fetched_at": datetime.now(timezone.utc).isoformat(),
    "event": event,
    "lobby": lobby,
    "matches": matches,
    "markets": all_markets,
}
(data_dir / "snapshot.json").write_text(json.dumps(snapshot, indent=2))
print(f"Saved to {data_dir / 'snapshot.json'}")

In [ ]:
# Replace team names with {team} so identical question types group together
team_names = {t for name in matches_df["name"] for t in name.split(" vs ")}

def to_template(question: str) -> str:
    for team in team_names:
        question = question.replace(team, "{team}")
    return question

template_counts = Counter(to_template(m["question"]) for m in all_markets)
pd.DataFrame(template_counts.most_common(), columns=["question template", "count"])

## 5. Submitting predictions — `POST /predictions/batch`

⚠️ **These cells place real predictions** on your bot's leaderboard entry.
That is safe to try: predictions can be revised until kickoff, and only the value at
market close gets scored. Flip the flag below to run the demo.

Rules to remember:
- `probability` is an **integer 1–99** (0 and 100 are rejected; 1 and 99 are the strongest stances).
- **One prediction per market** — a second POST returns `409 Conflict`; revise with PATCH instead.
- Batch accepts **≤50 per request** and validates each entry independently: one failure
  doesn't roll back the rest (per-entry errors come back in `results[].error`).
- Prefer batch over single `POST /predictions` — it keeps you under the 60 req/min limit.

In [ ]:
RUN_LIVE_DEMO = False  # <-- set to True to actually submit

if RUN_LIVE_DEMO:
    # Demo: one neutral 50% prediction on the first market of the next match.
    # The same call shape scales to 50 predictions at a time.
    demo_market = markets[0]
    results = client.submit_batch([
        {"market_id": demo_market["id"], "lobby_id": lobby["id"], "probability": 50},
    ])
    print(json.dumps(results, indent=2))
else:
    print("Demo skipped — set RUN_LIVE_DEMO = True to submit a real prediction.")

## 6. Updating a prediction — `PATCH /predictions/{id}`

Revise any prediction whose market hasn't closed. This is how the model will react to
team news, lineups, or improved estimates right up to kickoff.

In [ ]:
if RUN_LIVE_DEMO:
    prediction_id = results[0]["trade"]["id"]  # id returned by the submit above
    updated = client.update_prediction(prediction_id, probability=55)
    print(json.dumps(updated, indent=2))
else:
    print("Demo skipped — requires the prediction submitted in section 5.")

## 7. Monitoring — `GET /predictions` and `GET /results`

- `GET /predictions` — everything you've submitted. Note: `probability` is returned as a
  **0–1 decimal** (the stored form), even though you *write* integers 1–99. Multiply by 100 to compare.
  `brier_score` is `null` until the market settles.
- `GET /results` — settled predictions only, each with its final `brier_score`.
  This is our ground truth for tracking calibration as the tournament progresses
  (no webhooks exist — we poll this).

In [ ]:
my_predictions = client.predictions(lobby_id=lobby["id"])
print(f"Open + settled predictions: {len(my_predictions)}")
if my_predictions:
    display(pd.DataFrame(my_predictions)[["question", "probability", "market_status", "brier_score"]])

settled = client.results(lobby_id=lobby["id"])
print(f"Settled predictions: {len(settled)}")
if settled:
    settled_df = pd.DataFrame(settled)
    display(settled_df[["question", "probability_submitted", "brier_score"]])
    print(f"Mean Brier so far: {settled_df['brier_score'].mean():.4f}  (0 = perfect, 0.25 = always-50% baseline)")

## 8. Error handling & limits — reference

| Status | Meaning | What we do |
|---|---|---|
| `400` | Malformed request / empty batch / >50 entries | Fix the payload |
| `401` | Missing or invalid API key | Check `.env` |
| `403` | Valid key but not allowed (e.g. not a lobby member) | Join the lobby first |
| `404` | Lobby / market / prediction not found | Re-fetch ids |
| `409` | Already predicted on this market / already joined | Use PATCH instead of POST |
| `422` | Probability outside 1–99, malformed UUID | Clamp model output to [1, 99] |
| `429` | Rate limit (60 req/min/IP) | Client backs off and retries automatically |
| `500` | Server error | Client retries with exponential backoff |

Other operational notes from the docs:
- Each bot's API key = one leaderboard entry (max **2 bots** per account). Keys never expire and can't be self-revoked.
- Leaderboard caches up to 1 hour and only counts **settled** markets.
- The API is in beta — no SLA, so the model should tolerate downtime and retry.

## Next steps

1. **Model**: derive match-level outcome probabilities (Elo ratings / bookmaker odds, which are
   well-calibrated) and map them onto each question template from section 4b.
2. **Submit early, refine often**: predictions are editable until kickoff — submit a baseline
   for every open market, then PATCH as information (lineups, news) arrives.
3. **Track calibration**: poll `GET /results` and compare our Brier to the 0.25 coin-flip baseline;
   remember the leaderboard metric is *relative to the crowd*, with knockout 2× / final 3× weights.